In [1]:
# 📦 1. Importar librerías
import pandas as pd

In [2]:
# 📄 2. Cargar datasets
import os

drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Data/'
filename = 'sell-in.txt'
filepath = os.path.join(drive_base_path, filename)
df_sellin = pd.read_csv(filepath, sep='\t')
print(df_sellin.head(10))

filename = 'tb_productos.txt'
filepath = os.path.join(drive_base_path, filename)
df_productos = pd.read_csv(filepath, sep='\t')

filename = 'product_id_apredecir201912.txt'
filepath = os.path.join(drive_base_path, filename)
df_a_predecir = pd.read_csv(filepath, sep='\t')

   periodo  customer_id  product_id  plan_precios_cuidados  cust_request_qty  \
0   201701        10234       20524                      0                 2   
1   201701        10032       20524                      0                 1   
2   201701        10217       20524                      0                 1   
3   201701        10125       20524                      0                 1   
4   201701        10012       20524                      0                11   
5   201701        10080       20524                      0                 1   
6   201701        10015       20524                      0                 4   
7   201701        10062       20524                      0                 1   
8   201701        10159       20524                      0                 3   
9   201701        10183       20524                      0                 1   

   cust_request_tn       tn  
0          0.05300  0.05300  
1          0.13628  0.13628  
2          0.03028  0.03028  

In [3]:
# 📄 Leer lista de productos a predecir
with open(filepath, "r") as f:
    product_ids = [int(line.strip()) for line in f if line.strip().isdigit()]
    
print(product_ids)

[20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010, 20011, 20012, 20013, 20014, 20015, 20016, 20017, 20018, 20019, 20020, 20021, 20022, 20023, 20024, 20025, 20026, 20027, 20028, 20029, 20030, 20031, 20032, 20033, 20035, 20037, 20038, 20039, 20041, 20042, 20043, 20044, 20045, 20046, 20047, 20049, 20050, 20051, 20052, 20053, 20054, 20055, 20056, 20057, 20058, 20059, 20061, 20062, 20063, 20065, 20066, 20067, 20068, 20069, 20070, 20071, 20072, 20073, 20074, 20075, 20076, 20077, 20079, 20080, 20081, 20082, 20084, 20085, 20086, 20087, 20089, 20090, 20091, 20092, 20093, 20094, 20095, 20096, 20097, 20099, 20100, 20101, 20102, 20103, 20106, 20107, 20108, 20109, 20111, 20112, 20114, 20116, 20117, 20118, 20119, 20120, 20121, 20122, 20123, 20124, 20125, 20126, 20127, 20129, 20130, 20132, 20133, 20134, 20135, 20137, 20138, 20139, 20140, 20142, 20143, 20144, 20145, 20146, 20148, 20150, 20151, 20152, 20153, 20155, 20157, 20158, 20159, 20160, 20161, 20162, 20164, 20166, 20167, 20168

In [4]:
# 🧹 3. Preprocesamiento
# Convertir periodo a datetime
df_sellin['timestamp'] = pd.to_datetime(df_sellin['periodo'], format='%Y%m')

# Filtrar por año 2018 (201801 a 201812) y productos requeridos
df_filtered = df_sellin[
    (df_sellin['periodo'] >= 201801) & 
    (df_sellin['periodo'] <= 201812) &
    (df_sellin['product_id'].isin(product_ids))
]

In [5]:
# Agregar tn por periodo, cliente y producto
df_grouped = df_filtered.groupby(['timestamp', 'customer_id', 'product_id'], as_index=False)['tn'].sum()

In [6]:
# Agregar tn total por periodo y producto
df_monthly_product = df_grouped.groupby(['timestamp', 'product_id'], as_index=False)['tn'].sum()

## 🕒 Dynamic Time Warping (DTW) para predicciones

DTW puede mejorar las predicciones mediante:

1. **Encontrar productos similares** con patrones desalineados temporalmente
2. **Manejo de estacionalidades irregulares** (patrones "estirados" o "comprimidos")
3. **Robustez ante datos faltantes** o históricos de diferente longitud
4. **Identificación de templates** - productos con patrones estables como referencia
5. **Predicción por analogía** - usar patrones similares históricos para predecir

In [7]:
# 🔧 INSTALACIÓN Y CONFIGURACIÓN DTW
print("=== CONFIGURANDO DYNAMIC TIME WARPING ===")

# Instalar dtaidistance si es necesario
try:
    from dtaidistance import dtw
    from dtaidistance.dtw import distance_matrix_fast
    print("✅ dtaidistance ya instalado")
except ImportError:
    print("📦 Instalando dtaidistance...")
    %pip install dtaidistance
    from dtaidistance import dtw
    from dtaidistance.dtw import distance_matrix_fast

import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt

print("✅ Librerías DTW importadas correctamente")

=== CONFIGURANDO DYNAMIC TIME WARPING ===
✅ dtaidistance ya instalado
✅ Librerías DTW importadas correctamente
✅ Librerías DTW importadas correctamente


In [8]:
# 📊 PREPARAR DATOS PARA ANÁLISIS DTW
print("=== PREPARACIÓN DE SERIES TEMPORALES PARA DTW ===")

# Crear matriz de series temporales por producto (12 meses del año 2018)
df_dtw_base = df_monthly_product[df_monthly_product['timestamp'] >= '2018-01-01'].copy()
df_dtw_base = df_dtw_base.sort_values(['product_id', 'timestamp'])

# Crear pivot table: productos vs períodos
ts_matrix = df_dtw_base.pivot(index='product_id', columns='timestamp', values='tn')
ts_matrix = ts_matrix.fillna(0)  # Rellenar faltantes con 0

print(f"Matriz de series temporales: {ts_matrix.shape}")
print(f"Products: {ts_matrix.shape[0]}, Períodos: {ts_matrix.shape[1]}")
print(f"Períodos disponibles: {ts_matrix.columns.min()} a {ts_matrix.columns.max()}")

# Mostrar algunas series
print(f"\n=== PRIMERAS SERIES TEMPORALES ===")
print(ts_matrix.head())

# Normalizar series temporales para DTW (importante para comparación justa)
scaler_dtw = MinMaxScaler()
ts_matrix_norm = pd.DataFrame(
    scaler_dtw.fit_transform(ts_matrix.T).T,
    index=ts_matrix.index,
    columns=ts_matrix.columns
)

print(f"\n=== ESTADÍSTICAS POST-NORMALIZACIÓN ===")
print(f"Min global: {ts_matrix_norm.min().min():.3f}")
print(f"Max global: {ts_matrix_norm.max().max():.3f}")
print(f"Series con varianza > 0: {(ts_matrix_norm.var(axis=1) > 0.01).sum()}")

# Filtrar productos con suficiente variabilidad
productos_activos = ts_matrix_norm[ts_matrix_norm.var(axis=1) > 0.01].index.tolist()
print(f"Productos activos para DTW: {len(productos_activos)}")

ts_matrix_active = ts_matrix_norm.loc[productos_activos]

=== PREPARACIÓN DE SERIES TEMPORALES PARA DTW ===
Matriz de series temporales: (656, 12)
Products: 656, Períodos: 12
Períodos disponibles: 2018-01-01 00:00:00 a 2018-12-01 00:00:00

=== PRIMERAS SERIES TEMPORALES ===
timestamp   2018-01-01  2018-02-01  2018-03-01  2018-04-01  2018-05-01  \
product_id                                                               
20001       1169.07532  1043.76470  1856.83534  1251.28462  1293.89788   
20002        984.80167   712.00087   966.86044   999.20934  1103.39191   
20003        907.56304   788.30749   778.55594   765.47838   784.35885   
20004        415.52538   503.65326   488.92473   611.51237   641.37063   
20005        417.53208   399.20878   559.98671   496.41774   637.11135   

timestamp   2018-06-01  2018-07-01  2018-08-01  2018-09-01  2018-10-01  \
product_id                                                               
20001       1150.79169  1470.41009  1800.96168  1438.67455  2295.19832   
20002       1033.82845   977.40239  1161.8

In [9]:
# 🔄 CALCULAR MATRIZ DE DISTANCIAS DTW
print("=== CALCULANDO DISTANCIAS DTW ===")

# Convertir a numpy arrays para DTW
series_arrays = [ts_matrix_active.iloc[i].values for i in range(len(ts_matrix_active))]

# Calcular matriz de distancias DTW (puede tomar tiempo con muchos productos)
print(f"Calculando DTW para {len(series_arrays)} productos...")

# Para eficiencia, limitamos a los primeros N productos si hay demasiados (total = 780)
max_products = min(780, len(series_arrays))  # Limitar para demo
if len(series_arrays) > max_products:
    print(f"⚠️  Limitando análisis a {max_products} productos por eficiencia")
    series_arrays = series_arrays[:max_products]
    productos_dtw = productos_activos[:max_products]
else:
    productos_dtw = productos_activos

# Calcular matriz de distancias DTW
print("Calculando matriz de distancias... (puede tomar unos minutos)")
try:
    distance_matrix = distance_matrix_fast(series_arrays)
    print(f"✅ Matriz DTW calculada: {distance_matrix.shape}")
except Exception as e:
    print(f"⚠️  Error con función rápida, usando método estándar: {e}")
    # Fallback a método manual
    n = len(series_arrays)
    distance_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            dist = dtw.distance(series_arrays[i], series_arrays[j])
            distance_matrix[i][j] = dist
            distance_matrix[j][i] = dist
    print(f"✅ Matriz DTW calculada (método manual): {distance_matrix.shape}")

# Estadísticas de distancias
print(f"\n=== ESTADÍSTICAS DISTANCIAS DTW ===")
print(f"Distancia mínima: {distance_matrix[distance_matrix > 0].min():.3f}")
print(f"Distancia máxima: {distance_matrix.max():.3f}")
print(f"Distancia promedio: {distance_matrix[distance_matrix > 0].mean():.3f}")
print(f"Desviación estándar: {distance_matrix[distance_matrix > 0].std():.3f}")

=== CALCULANDO DISTANCIAS DTW ===
Calculando DTW para 656 productos...
Calculando matriz de distancias... (puede tomar unos minutos)
✅ Matriz DTW calculada: (656, 656)

=== ESTADÍSTICAS DISTANCIAS DTW ===
Distancia mínima: 0.003
Distancia máxima: 2.710
Distancia promedio: 0.968
Desviación estándar: 0.357
✅ Matriz DTW calculada: (656, 656)

=== ESTADÍSTICAS DISTANCIAS DTW ===
Distancia mínima: 0.003
Distancia máxima: 2.710
Distancia promedio: 0.968
Desviación estándar: 0.357


In [10]:
# 🎯 CLUSTERING BASADO EN DTW
print("=== CLUSTERING DE PRODUCTOS SIMILARES ===")

# Usar clustering aglomerativo con distancias DTW
n_clusters = min(50, len(productos_dtw) // 3)  # Número adaptativo de clusters
clustering = AgglomerativeClustering(
    n_clusters=n_clusters,
    linkage='average',
    metric='precomputed'
)

clusters = clustering.fit_predict(distance_matrix)

# Crear DataFrame con resultados
df_clusters = pd.DataFrame({
    'product_id': productos_dtw,
    'cluster': clusters
})

print(f"Productos agrupados en {n_clusters} clusters:")
cluster_counts = df_clusters['cluster'].value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    print(f"  Cluster {cluster_id}: {count} productos")

# Encontrar productos más similares para cada producto
print(f"\n=== PRODUCTOS MÁS SIMILARES (TOP 3) ===")
df_similares = []

for i, product_id in enumerate(productos_dtw):
    # Obtener distancias de este producto a todos los demás
    distancias = distance_matrix[i].copy()
    distancias[i] = np.inf  # Excluir el producto mismo
    
    # Encontrar los 3 más similares (menor distancia)
    indices_similares = np.argsort(distancias)[:3]
    
    for rank, idx in enumerate(indices_similares, 1):
        df_similares.append({
            'product_id': product_id,
            'similar_product': productos_dtw[idx],
            'dtw_distance': distancias[idx],
            'similarity_rank': rank,
            'cluster_origen': clusters[i],
            'cluster_similar': clusters[idx]
        })

df_similares = pd.DataFrame(df_similares)

# Mostrar algunos ejemplos
print("Ejemplos de productos similares:")
for product in productos_dtw[:5]:
    similares = df_similares[df_similares['product_id'] == product]
    print(f"\nProducto {product}:")
    for _, row in similares.iterrows():
        print(f"  #{row['similarity_rank']}: Producto {row['similar_product']} "
              f"(distancia: {row['dtw_distance']:.3f}, cluster: {row['cluster_similar']})")

=== CLUSTERING DE PRODUCTOS SIMILARES ===
Productos agrupados en 50 clusters:
  Cluster 0: 12 productos
  Cluster 1: 5 productos
  Cluster 2: 41 productos
  Cluster 3: 7 productos
  Cluster 4: 68 productos
  Cluster 5: 20 productos
  Cluster 6: 8 productos
  Cluster 7: 44 productos
  Cluster 8: 13 productos
  Cluster 9: 7 productos
  Cluster 10: 173 productos
  Cluster 11: 21 productos
  Cluster 12: 4 productos
  Cluster 13: 10 productos
  Cluster 14: 3 productos
  Cluster 15: 3 productos
  Cluster 16: 4 productos
  Cluster 17: 10 productos
  Cluster 18: 4 productos
  Cluster 19: 19 productos
  Cluster 20: 8 productos
  Cluster 21: 9 productos
  Cluster 22: 8 productos
  Cluster 23: 9 productos
  Cluster 24: 2 productos
  Cluster 25: 1 productos
  Cluster 26: 5 productos
  Cluster 27: 1 productos
  Cluster 28: 1 productos
  Cluster 29: 4 productos
  Cluster 30: 5 productos
  Cluster 31: 3 productos
  Cluster 32: 1 productos
  Cluster 33: 2 productos
  Cluster 34: 2 productos
  Cluster 

In [11]:
# EXPORTAR RESULTADOS DTW
output_dir = "C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Dinamic Time Warping (DTW)"
os.makedirs(output_dir, exist_ok=True)

# Exportar resultado DTW
df_similares.to_csv(os.path.join(output_dir, "productos_similares_dtw_2018_n50.csv"), index=False)
df_clusters.to_csv(os.path.join(output_dir, "clusters_dtw_2018_n50.csv"), index=False)